In [0]:
import os
import zipfile
import shutil

# 1. Fill in your Kaggle credentials
os.environ['KAGGLE_USERNAME'] = "kaggke_token_databricks"
os.environ['KAGGLE_KEY'] = "KGAT_87287d8faf5dffeedb5ec71c4e713159"

# 2. Create the UC volume (os.makedirs cannot create volumes — use SQL DDL)
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.raw_data")

# 3. Setup paths
volume_path = "/Volumes/workspace/default/raw_data"
download_dir = "/tmp/instacart_download"

os.makedirs(download_dir, exist_ok=True)

# 4. Pull dataset directly to cluster storage
%pip install --quiet kaggle
!kaggle datasets download -d yasserh/instacart-online-grocery-basket-analysis-dataset -p {download_dir}

# 4. Extract main archive into the Volume
for item in os.listdir(download_dir):
    if item.endswith(".zip"):
        zip_path = os.path.join(download_dir, item)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(volume_path)

# 5. Extract any nested .csv.zip files if present
for item in os.listdir(volume_path):
    if item.endswith(".zip"):
        inner_zip = os.path.join(volume_path, item)
        with zipfile.ZipFile(inner_zip, 'r') as zf:
            zf.extractall(volume_path)
        os.remove(inner_zip)

# 6. Clear cluster temp space
shutil.rmtree(download_dir)
print("Data download and extraction complete!")

In [0]:
%restart_python